In [3]:
cd ..

/home/serafeim/Desktop/bet


In [4]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from prepare_data_inference import prepare_data_inference
from pyspark.ml.functions import vector_to_array
import mlflow 


In [5]:
def compute_metrics(df, threshold):
    pred = df.withColumn(
        "pred_label",
        (F.col("p_churn") >= threshold).cast("int")
    )
    metrics = pred.groupBy("next_7d_churn_idx", "pred_label").count()
    tp = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 1").select("count").first()
    fp = metrics.filter("next_7d_churn_idx = 0 AND pred_label = 1").select("count").first()
    fn = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 0").select("count").first()

    tp = tp[0] if tp else 0
    fp = fp[0] if fp else 0
    fn = fn[0] if fn else 0

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return precision, recall, f1

In [6]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 10:25:19 WARN Utils: Your hostname, serafeim-virtual-machine, resolves to a loopback address: 127.0.1.1; using 10.100.2.34 instead (on interface ens192)
26/03/05 10:25:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 10:25:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

In [8]:
test_date = '2024-05-20'
data_inference = prepare_data_inference(test_date)
data_inference.show(3)

failed_withdrawals_30d
deposit_count_30d
withdrawal_count_30d
withdrawal_ratio
+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|   balance_30d_ago|    balance_7d_ago|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|
+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|       964|              243.12|              1813

In [9]:
def compare(df1,df2): 
   assert (( df1.exceptAll(df2).count() == 0) & (df2.exceptAll(df1).count() == 0))

m1 = player_behavior.filter(F.col('reference_date')==test_date).select('player_idx').join(data_inference, how='inner', on='player_idx', )
compare(m1, data_inference)

In [22]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

mlflow.set_experiment("batch-mode experiment")

model_run_id = '8d59c351ba2d4092884b2dcffcb80d76'
run = mlflow.get_run(model_run_id)
model_uri = f'runs:/{model_run_id}/spark_model'
loaded_model = mlflow.spark.load_model(model_uri)

test_start = run.data.params["test_start"]
test_end = run.data.params["test_end"]
test_df =  (player_behavior.filter(F.col('reference_date')>=test_start)
        .join(player_snapshot, on="player_idx", how="left")
        .join(labels, on=["player_idx", "reference_date"], how="inner")
         .withColumn("next_7d_churn_idx", F.col("next_7d_churn").cast("int")))

print(test_start, test_end)

data_inference_ml = (player_behavior.select('player_idx','reference_date').filter(F.col('reference_date')==test_date)
.join(data_inference, how ='inner', on='player_idx')
.join(player_snapshot, on="player_idx", how="inner")
)

with mlflow.start_run(run_name=test_date):
    test_preds = (loaded_model.transform(data_inference_ml)
    .withColumn("p_churn", vector_to_array("probability")[1]))

    



2026/03/05 12:14:39 INFO mlflow.spark: URI 'runs:/8d59c351ba2d4092884b2dcffcb80d76/spark_model/sparkml' does not point to the current DFS.
2026/03/05 12:14:39 INFO mlflow.spark: File 'runs:/8d59c351ba2d4092884b2dcffcb80d76/spark_model/sparkml' not found on DFS. Will attempt to upload the file.


2024-06-01 2024-06-22


In [23]:
def result_display(preds):
    preds = preds.select('player_idx','reference_date', 'p_churn', 'prediction')
    preds = preds.withColumn('risk_level', 
        F.when(F.col('p_churn')>=0.8, 'High')
        .when(F.col('p_churn')>=0.6, 'Medium')
        .when(F.col('p_churn')>=0.4, 'Low')
        .otherwise(F.lit('None')))
    return preds

In [24]:
results = result_display(test_preds)

In [25]:
results.show(4)

+----------+--------------+-------------------+----------+----------+
|player_idx|reference_date|            p_churn|prediction|risk_level|
+----------+--------------+-------------------+----------+----------+
|         0|    2024-05-20| 0.6054528994186144|       0.0|    Medium|
|         1|    2024-05-20| 0.4862471576902906|       0.0|       Low|
|         2|    2024-05-20|0.14583529557323394|       0.0|      None|
|         3|    2024-05-20|0.49047382167614717|       0.0|       Low|
+----------+--------------+-------------------+----------+----------+
only showing top 4 rows


In [26]:
dates = (
    test_df
    .select("reference_date")
    .distinct()
    .orderBy("reference_date")
    .collect()
)

dates = [r.reference_date for r in dates]

In [27]:

results = []
threshold = 0.8

for d in dates:

    daily_df = test_df.filter(F.col("reference_date") == d)

    preds = (
         loaded_model.transform(daily_df)
        .withColumn("p_churn", vector_to_array("probability")[1])
    )

    precision, recall, f1 = compute_metrics(preds, threshold)

    results.append({
        "date": str(d),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "num_players": preds.count(),
        "num_flagged": preds.filter(F.col("p_churn") >= threshold).count()
    })

In [30]:

backtest_df = spark.createDataFrame(results)
backtest_df.orderBy("date").show()

+----------+------------------+-----------+-----------+------------------+-------------------+
|      date|                f1|num_flagged|num_players|         precision|             recall|
+----------+------------------+-----------+-----------+------------------+-------------------+
|2024-06-01|0.7200000000000001|         44|        651|0.8181818181818182| 0.6428571428571429|
|2024-06-02|0.7027027027027027|         47|        653|0.8297872340425532|           0.609375|
|2024-06-03|0.7192982456140352|         47|        654|0.8723404255319149| 0.6119402985074627|
|2024-06-04|0.6833333333333332|         48|        654|0.8541666666666666| 0.5694444444444444|
|2024-06-05|0.6504065040650406|         48|        654|0.8333333333333334| 0.5333333333333333|
|2024-06-06|             0.624|         47|        654|0.8297872340425532|                0.5|
|2024-06-07|0.6016260162601625|         46|        654|0.8043478260869565| 0.4805194805194805|
|2024-06-08|0.5669291338582676|         46|       

In [ ]:
#churn rate per risk level
backtest_df.groupBy()

+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+----------------+---------------+------------------+-----------------+--------------------+-------------------+-------------+-----------------+-----------+--------------+-------------+--------------+--------------------+-----------------------+--------------------+--------------------+--------------------+----------+--------------------+----------+
|player_idx|reference_date|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|fai